# 8. Transformer Decoder

**Цель:** Реализовать декодерный блок трансформера: Masked Self-Attention → Cross-Attention → FFN. Понять каузальную маскировку и cross-attention.

---

In [2]:
import sys, os, logging, math

import torch  # Основной фреймворк
import torch.nn as nn  # Нейросетевые слои
import torch.nn.functional as F  # Функции (softmax, gelu)
import numpy as np  # Численные операции
import matplotlib.pyplot as plt  # Визуализация

if torch.cuda.is_available():  # GPU NVIDIA
    device = torch.device("cuda")

elif torch.backends.mps.is_available():  # GPU Apple
    device = torch.device("mps")

else:  # CPU
    device = torch.device("cpu")

## 8.1 Каузальная (Causal) маскировка

**Зачем?**
- В декодере каждый токен может "смотреть" только на предыдущие токены
- Нельзя заглядывать в будущее при авторегрессивной генерации

**Реализация:**
- Верхнетреугольная матрица с -inf
- scores[i, j] = -inf для всех j > i (будущие позиции)
- После softmax внимание на будущие позиции = 0

In [4]:

def create_causal_mask(seq_len):  # Создаёт верхнетреугольную маску
    """Создаёт каузальную маску (True = маскировать)."""
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()  # True для j > i (будущие токены)
    return mask

seq_len = 6
causal_mask = create_causal_mask(seq_len)

print("Causal mask (True = masked):")
print(causal_mask.numpy())

# Демонстрация эффекта на attention
d_k = 8
Q = torch.randn(1, seq_len, d_k)  # Случайные запросы
K = torch.randn(1, seq_len, d_k)  # Случайные ключи
V = torch.randn(1, seq_len, d_k)  # Случайные значения

scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # Скалярные произведения до маски
scores_masked = scores.masked_fill(causal_mask.unsqueeze(0), float('-inf'))  # Применяем каузальную маску
attn = F.softmax(scores_masked, dim=-1)  # После маски внимание на будущее = 0

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(scores[0].detach().numpy(), cmap='RdBu')
axes[0].set_title('Scores (before mask)')
axes[0].set_xlabel('Key')
axes[0].set_ylabel('Query')
plt.colorbar(axes[0].images[0], ax=axes[0], fraction=0.046)

axes[1].imshow(attn[0].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Attention (after causal mask)')
axes[1].set_xlabel('Key')
axes[1].set_ylabel('Query')
plt.colorbar(axes[1].images[0], ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

Causal mask (True = masked):
[[False  True  True  True  True  True]
 [False False  True  True  True  True]
 [False False False  True  True  True]
 [False False False False  True  True]
 [False False False False False  True]
 [False False False False False False]]


## 8.2 MultiHeadAttention (c поддержкой каузальной маски)

Расширяем MHA поддержкой каузальной маски.

In [6]:

class MultiHeadAttention(nn.Module):  # Многоголовое внимание с поддержкой causal
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0  # d_model кратен n_heads
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # Размерность одной головы
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, Q, K, V, mask=None, causal=False):  # causal=True — каузальная маскировка
        batch = Q.size(0)
        Q = self.W_Q(Q).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if causal:  # Каузальная маска (запрет на будущие токены)
            seq_len = Q.size(-2)
            causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=Q.device), diagonal=1).bool()  # Верхний треугольник
            scores = scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn = self.dropout(F.softmax(scores, dim=-1))
        output = torch.matmul(attn, V).transpose(1, 2).contiguous().view(batch, -1, self.d_model)
        return self.W_O(output), attn

mha = MultiHeadAttention(d_model=32, n_heads=4)  # Создаём MHA с поддержкой causal
x = torch.randn(2, 10, 32)
out, attn = mha(x, x, x, causal=True)  # Self-attention с каузальной маской

## 8.3 FeedForward (из предыдущего ноутбука)

In [8]:
class FeedForward(nn.Module):  # Двухслойная FFN (из ноутбука 07)
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model  # По умолчанию 4×d_model
        self.fc1 = nn.Linear(d_model, d_ff)  # Расширение
        self.fc2 = nn.Linear(d_ff, d_model)  # Сжатие
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.fc2(self.dropout(F.gelu(self.fc1(x))))  # Linear → GELU → Dropout → Linear

ffn = FeedForward(32)

## 8.4 TransformerDecoderBlock

**Три под-слоя (против двух в энкодере):**
1. Masked Self-Attention (каузальный) → Add & Norm
2. Cross-Attention (Q от декодера, K, V от энкодера) → Add & Norm
3. Feed-Forward Network → Add & Norm

In [10]:

class TransformerDecoderBlock(nn.Module):  # Один блок декодера (3 под-слоя)
    """Один блок декодера трансформера."""
    
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        # Self-attention (causal)
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)  # 1. Masked Self-Attention
        self.norm1 = nn.LayerNorm(d_model)  # Norm после self-attention
        self.dropout1 = nn.Dropout(dropout)
        
        # Cross-attention
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout)  # 2. Cross-Attention
        self.norm2 = nn.LayerNorm(d_model)  # Norm после cross-attention
        self.dropout2 = nn.Dropout(dropout)
        
        # FFN
        self.ffn = FeedForward(d_model, d_ff, dropout)  # 3. Feed-Forward Network
        self.norm3 = nn.LayerNorm(d_model)  # Norm после FFN
        self.dropout3 = nn.Dropout(dropout)
        
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):  # x — декодер, encoder_output — энкодер
        # 1. Masked Self-Attention (каузальный)
        attn_out, self_attn = self.self_attention(self.norm1(x), self.norm1(x), self.norm1(x),
                                                  mask=tgt_mask, causal=True)
        x = x + self.dropout1(attn_out)  # Остаточная связь
        
        # 2. Cross-Attention: Q от декодера, K,V от энкодера
        cross_out, cross_attn = self.cross_attention(self.norm2(x), self.norm2(encoder_output), self.norm2(encoder_output), mask=src_mask)
        x = x + self.dropout2(cross_out)  # Остаточная связь после cross-attention
        
        # 3. Feed-Forward Network
        ffn_out = self.ffn(self.norm3(x))  # FFN с Pre-Norm
        x = x + self.dropout3(ffn_out)  # Остаточная связь после FFN
        
        return x, self_attn, cross_attn  # Выход + веса обоих вниманий

decoder_block = TransformerDecoderBlock(d_model=32, n_heads=4)  # Создаём блок декодера

In [11]:

batch, seq_len_dec, seq_len_enc, d_model = 2, 8, 12, 32  # Размерности: декодер 8 токенов, энкодер 12
batch, seq_len_dec, seq_len_enc, d_model = 2, 8, 12, 32  # Размерности: декодер 8 токенов, энкодер 12
encoder_out = torch.randn(batch, seq_len_enc, d_model)

output, self_attn, cross_attn = decoder_block(x, encoder_out)

print(f"Decoder input:        {x.shape}")
print(f"Encoder output:       {encoder_out.shape}")
print(f"Decoder output:       {output.shape}")
print(f"Self-attention:       {self_attn.shape}")
print(f"Cross-attention:      {cross_attn.shape}")

Decoder input:        torch.Size([2, 8, 32])
Encoder output:       torch.Size([2, 12, 32])
Decoder output:       torch.Size([2, 8, 32])
Self-attention:       torch.Size([2, 4, 8, 8])
Cross-attention:      torch.Size([2, 4, 8, 12])


## 8.5 Стекирование: TransformerDecoder

In [13]:

class TransformerDecoder(nn.Module):  # Стопка декодерных блоков
    def __init__(self, num_layers, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([  # Создаём num_layers блоков
            TransformerDecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):  # x — целевая последовательность
        all_self_attns = []  # Self-attention со всех слоёв
        all_cross_attns = []  # Cross-attention со всех слоёв
        for i, layer in enumerate(self.layers):
            x, self_attn, cross_attn = layer(x, encoder_output, src_mask, tgt_mask)
            all_self_attns.append(self_attn)
            all_cross_attns.append(cross_attn)
        return x, all_self_attns, all_cross_attns

decoder = TransformerDecoder(num_layers=4, d_model=32, n_heads=4)  # 4 слоя декодера
output, self_attns, cross_attns = decoder(x, encoder_out)
print(f"Decoder output:     {output.shape}")
print(f"Self-attention maps: {len(self_attns)}")
print(f"Cross-attention maps: {len(cross_attns)}")

Decoder output:     torch.Size([2, 8, 32])
Self-attention maps: 4
Cross-attention maps: 4


## 8.6 Визуализация: Causal Mask в действии

Сравниваем self-attention (causal) vs cross-attention.

In [15]:

x = torch.randn(1, 6, 32)  # Декодер: 6 токенов
enc_out = torch.randn(1, 10, 32)  # Энкодер: 10 токенов

out, self_attns, cross_attns = decoder(x, enc_out)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i in range(4):
    # Self-attention (causal), head 0
    im = axes[0, i].imshow(self_attns[i][0, 0].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
    axes[0, i].set_title(f'Layer {i+1}: Self-Attn (causal)')
    axes[0, i].set_xlabel('Key (decoder)')
    axes[0, i].set_ylabel('Query (decoder)')
    plt.colorbar(im, ax=axes[0, i], fraction=0.046)
    
    # Cross-attention (no mask)
    im = axes[1, i].imshow(cross_attns[i][0, 0].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
    axes[1, i].set_title(f'Layer {i+1}: Cross-Attn')
    axes[1, i].set_xlabel('Key (encoder)')
    axes[1, i].set_ylabel('Query (decoder)')
    plt.colorbar(im, ax=axes[1, i], fraction=0.046)

plt.suptitle('Decoder Attention Patterns: Causal Self-Attention vs Cross-Attention')
plt.tight_layout()
plt.show()

In [ ]:

BOS, EOS, PAD = 0, 1, 2  # Специальные токены
vocab_size = 16
max_len = 10
d_model = 32

class MiniDecoderLM(nn.Module):  # Мини-декодер для авторегрессии
    def __init__(self, vocab_size, d_model, n_heads, num_layers, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)  # Эмбеддинги токенов
        pe = torch.zeros(max_len, d_model)  # Синусоидальные PE
        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)
        self.decoder = TransformerDecoder(num_layers, d_model, n_heads)  # Декодер
        self.output_proj = nn.Linear(d_model, vocab_size)  # Выходная проекция на словарь
        for p in self.parameters():  # Инициализация Xavier
            if p.dim() > 1:  # Только для матриц (не bias)
                nn.init.xavier_uniform_(p)  # Xavier Glorot

    def forward(self, x, encoder_output, src_mask=None):  # Прямой проход
        x = self.embedding(x) * math.sqrt(d_model) + self.pe[:, :x.size(1), :]  # Эмбеддинги + PE
        dec_out, _, _ = self.decoder(x, encoder_output, src_mask)
        return self.output_proj(dec_out)

model = MiniDecoderLM(vocab_size, d_model=32, n_heads=4, num_layers=3, max_len=max_len).to(device)  # Создаём модель

# Быстрое обучение на задаче копирования
def make_copy_data(num_samples):  # Создаёт данные для задачи копирования
    data = []
    for _ in range(num_samples):
        length = np.random.randint(2, max_len - 1)
        seq = [BOS] + np.random.randint(3, vocab_size, size=length).tolist() + [EOS]  # [BOS, токены..., EOS]
        seq = seq + [PAD] * (max_len - len(seq))  # Дополняем до max_len
        data.append(seq)
    return torch.tensor(data)

train_data = make_copy_data(1000)  # 1000 обучающих примеров
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Оптимизатор Adam
criterion = nn.CrossEntropyLoss(ignore_index=PAD)  # Функция потерь (PAD не учитывается)

for epoch in range(20):  # 20 эпох обучения
    perm = torch.randperm(len(train_data))  # Перемешиваем данные
    epoch_loss = 0
    for i in range(0, len(train_data), 64):  # Мини-батчи по 64
        idx = perm[i:i+64]
        batch = train_data[idx].to(device)
        encoder_out = torch.randn(batch.size(0), max_len, d_model, device=device)  # Фиктивный энкодер
        output = model(batch[:, :-1], encoder_out)  # Teacher forcing: предсказываем следующий токен
        loss = criterion(output.reshape(-1, vocab_size), batch[:, 1:].reshape(-1))  # Loss между предсказанием и сдвигом
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if epoch % 10 == 0:  # Печатаем каждые 10 эпох
        print(f"Epoch {epoch}: loss={epoch_loss:.4f}")

# Авторегрессивная генерация
@torch.no_grad()
def generate(model, prompt, max_new_tokens=10):  # Авторегрессивная генерация
    model.eval()  # Режим инференса (без Dropout)
    encoder_out = torch.randn(batch.size(0), max_len, d_model, device=device)  # Фиктивный энкодер
    for _ in range(max_new_tokens):  # Генерируем по одному токену
        logits = model(prompt, encoder_out)[:, -1, :]  # Логиты последнего токена
        next_token = logits.argmax(-1, keepdim=True)  # Жадный выбор самого вероятного токена
        prompt = torch.cat([prompt, next_token], dim=-1)  # Добавляем сгенерированный токен
        if (next_token == EOS).all():  # Останавливаемся при EOS
            break
    return prompt

prompt = torch.tensor([[BOS, 5, 12]], device=device)  # Начальный промпт
result = generate(model, prompt)
tokens = [t for t in result[0].tolist() if t not in (BOS, EOS, PAD)]  # Убираем спецтокены
print(f"Prompt: [BOS, 5, 12]")
print(f"Generated: {tokens}")

In [ ]:
print("=== Transformer Decoder complete ===")  # Итоговый вывод
print("Topics covered:")
print("  - Causal (masked) self-attention")
print("  - Cross-attention: Q from decoder, K,V from encoder")
print("  - TransformerDecoderBlock (3 sub-layers)")
print("  - Stacked TransformerDecoder")
print("  - Causal vs cross-attention visualization")
print("  - Autoregressive generation demo")